In [1]:
from pathlib import Path
from liffile import LifFile
import numpy as np
import tifffile

In [ ]:
number_to_change_by = 3
lif_path = Path(r"Z:\Bel\Farid\Permeability\Permeabilty 18.02.26.lif")  # <-- update this
output_dir = lif_path.parent / "cropped_tifs"
output_dir.mkdir(exist_ok=True)

lif_stem = lif_path.stem.replace("/", "_")

with LifFile(lif_path) as lif:
    for img in lif.images:
        name = "".join(img.path).replace("/", "_")
        dims = tuple(img.dims)

        try:
            data = img.asarray()
        except (KeyError, IndexError) as e:
            print(f"\n{name}: SKIP - could not read image data ({type(e).__name__}: {e})")
            continue

        print(f"\n{name}: dims={dims}, shape={data.shape}")

        if "T" not in dims or "Z" not in dims:
            print(f"  SKIP - missing T or Z axis")
            continue

        t_ax = dims.index("T")
        z_ax = dims.index("Z")
        n_t = data.shape[t_ax]
        n_z = data.shape[z_ax]

        if n_t != 3:
            print(f"  SKIP - expected 3 timepoints, got {n_t}")
            continue

        # Move T to axis-0 and Z to axis-1 for easy slicing
        axes_order = list(range(data.ndim))
        axes_order.remove(t_ax)
        axes_order.remove(z_ax)
        axes_order = [t_ax, z_ax] + axes_order
        arr = np.transpose(data, axes_order)
        # arr shape: (T, Z, <remaining dims e.g. C, Y, X or Y, X>)

        remaining_dims = [dims[i] for i in axes_order[2:]]
        print(f"  Z slices = {n_z}, remaining dims = {remaining_dims}")

        n_out = n_z - 2*number_to_change_by
        if n_out <= 0:
            print(f"  SKIP - too few Z slices ({n_z}) to crop")
            continue

        t0_crop = arr[0, 2*number_to_change_by:, ...]
        t1_crop = arr[1, number_to_change_by:n_z - number_to_change_by, ...]
        t2_crop = arr[2, :n_z - 2*number_to_change_by, ...]

        cropped = np.stack([t0_crop, t1_crop, t2_crop], axis=0)
        print(f"  Cropped shape (T, Z, ...): {cropped.shape}")

        # Ensure TZCYX order for ImageJ hyperstack
        has_c = "C" in remaining_dims
        if not has_c:
            # remaining_dims is [Y, X] -> insert C dimension
            cropped = cropped[:, :, np.newaxis, :, :]

        # Read pixel sizes from LIF metadata for ImageJ calibration
        metadata = {}
        resolution = None
        try:
            xa = img.asxarray()
            coords = xa.coords
            if "X" in coords and coords["X"].size >= 2:
                x_um = abs(float(coords["X"][1] - coords["X"][0])) * 1e6
                resolution = (1.0 / x_um, 1.0 / x_um)  # pixels per µm (X, Y)
                metadata["unit"] = "um"
                print(f"  XY pixel size = {x_um:.4f} µm  (resolution = {resolution[0]:.4f} px/µm)")
            if "Z" in coords and coords["Z"].size >= 2:
                z_um = abs(float(coords["Z"][1] - coords["Z"][0])) * 1e6
                metadata["spacing"] = z_um
                print(f"  Z step = {z_um:.4f} µm")
        except Exception as e:
            print(f"  [WARN] Could not read pixel sizes: {e}")

        out_path = output_dir / f"{lif_stem}__{name}_cropped.tif"
        tifffile.imwrite(
            str(out_path),
            cropped.astype(data.dtype),
            imagej=True,
            resolution=resolution,
            metadata=metadata,
        )
        print(f"  Saved -> {out_path}")

print("\nDone.")



valve 4_P 1: dims=('T', 'C', 'Z', 'Y', 'X'), shape=(3, 3, 47, 512, 512)
  Z slices = 47, remaining dims = ['C', 'Y', 'X']
  Cropped shape (T, Z, ...): (3, 41, 3, 512, 512)
  XY pixel size = 2.2750 µm  (resolution = 0.4396 px/µm)
  Z step = 5.0001 µm
  Saved -> Z:\Bel\Farid\Permeability\cropped_tifs\Permeabilty 18.02.26__valve 4_P 1_cropped.tif

valve 4_P 2: dims=('T', 'C', 'Z', 'Y', 'X'), shape=(3, 3, 47, 512, 512)
  Z slices = 47, remaining dims = ['C', 'Y', 'X']
  Cropped shape (T, Z, ...): (3, 41, 3, 512, 512)
  XY pixel size = 2.2750 µm  (resolution = 0.4396 px/µm)
  Z step = 5.0001 µm
  Saved -> Z:\Bel\Farid\Permeability\cropped_tifs\Permeabilty 18.02.26__valve 4_P 2_cropped.tif

valve 4_P 3: dims=('T', 'C', 'Z', 'Y', 'X'), shape=(3, 3, 47, 512, 512)
  Z slices = 47, remaining dims = ['C', 'Y', 'X']
  Cropped shape (T, Z, ...): (3, 41, 3, 512, 512)
  XY pixel size = 2.2750 µm  (resolution = 0.4396 px/µm)
  Z step = 5.0001 µm
  Saved -> Z:\Bel\Farid\Permeability\cropped_tifs\Perme